In [1]:
# Cell 4: Configure the full environment
import torch
import torch.nn as nn
import numpy as np

# ART imports
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod          # FGSM
from art.attacks.evasion import ProjectedGradientDescent    # PGD

# ── Device Selection (CUDA → MPS → CPU) ──────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
    device_label = "NVIDIA CUDA GPU"
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    device_label = "Apple Metal (MPS) GPU"
else:
    device = torch.device("cpu")
    device_label = "CPU (no GPU found)"

# ART does not fully support MPS — force its operations to CPU
art_device = "cpu" if device.type == "mps" else str(device)

# ── Reproducibility Seeds ─────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
elif device.type == "mps":
    torch.mps.manual_seed(SEED)

# ── Sanity Check ──────────────────────────────────────────────────────
test_tensor = torch.ones(3, 3).to(device)

print("=" * 40)
print("       Environment Ready!")
print("=" * 40)
print(f"  PyTorch version : {torch.__version__}")
print(f"  Training device : {device_label}")
print(f"  ART device      : {art_device} (ART MPS workaround)")
print(f"  Tensor device   : {test_tensor.device} ✅")
print(f"  Random seed     : {SEED}")
print("=" * 40)

       Environment Ready!
  PyTorch version : 2.9.1
  Training device : Apple Metal (MPS) GPU
  ART device      : cpu (ART MPS workaround)
  Tensor device   : mps:0 ✅
  Random seed     : 42


In [2]:
import pickle
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.optim as optim
from torchvision import models
import copy, time

In [3]:
CIFAR10_PATH = "./cifar-10-batches-py"

In [4]:
# ── Helper: unpickle a single batch file ─────────────────────────────
def unpickle(file):
    with open(file, "rb") as f:
        data = pickle.load(f, encoding="bytes")
    return data

In [5]:
# ── Load class names from batch.meta ─────────────────────────────────
meta       = unpickle(f"{CIFAR10_PATH}/batches.meta")
CIFAR10_CLASSES = [c.decode("utf-8") for c in meta[b"label_names"]]

In [6]:
# ── Load all 5 training batches ───────────────────────────────────────
train_images, train_labels = [], []

for i in range(1, 6):  # data_batch_1 ... data_batch_5
    batch = unpickle(f"{CIFAR10_PATH}/data_batch_{i}")
    train_images.append(batch[b"data"])
    train_labels.extend(batch[b"labels"])

# Stack into single arrays: shape → (50000, 3072)
train_images = np.vstack(train_images)
train_labels = np.array(train_labels)

In [7]:
# ── Load test batch ───────────────────────────────────────────────────
test_batch   = unpickle(f"{CIFAR10_PATH}/test_batch")
test_images  = test_batch[b"data"]                   # shape: (10000, 3072)
test_labels  = np.array(test_batch[b"labels"])

# ── Reshape from flat (N, 3072) → (N, 3, 32, 32) then → (N, 32, 32, 3)
# CIFAR stores in CHW format; we convert to HWC for transforms
train_images = train_images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
test_images  = test_images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

In [8]:
# ── Custom Dataset Class ──────────────────────────────────────────────
class CIFAR10Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images       # numpy (N, 32, 32, 3) uint8
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img   = self.images[idx]      # (32, 32, 3) uint8
        label = int(self.labels[idx])

        # Convert numpy HWC → PIL for torchvision transforms
        from PIL import Image
        img = Image.fromarray(img)

        if self.transform:
            img = self.transform(img)

        return img, label

In [9]:
# ── Normalization Constants ───────────────────────────────────────────
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

In [10]:
# ── Transforms ───────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),                            # → [0, 1]
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)  # → standardized
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

In [11]:
# ── Instantiate Datasets ──────────────────────────────────────────────
train_dataset = CIFAR10Dataset(train_images, train_labels, transform=train_transform)
test_dataset  = CIFAR10Dataset(test_images,  test_labels,  transform=test_transform)

In [12]:
# ── DataLoaders ───────────────────────────────────────────────────────
BATCH_SIZE  = 128
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

In [13]:
# ── Verification ──────────────────────────────────────────────────────
images, labels = next(iter(train_loader))

print("=" * 40)
print("     CIFAR-10 Dataset Ready!")
print("=" * 40)
print(f"  Training samples : {len(train_dataset):,}")
print(f"  Test samples     : {len(test_dataset):,}")
print(f"  Classes          : {CIFAR10_CLASSES}")
print("-" * 40)
print(f"  Batch shape      : {images.shape}")       # [128, 3, 32, 32]
print(f"  Pixel min        : {images.min():.4f}")
print(f"  Pixel max        : {images.max():.4f}")
print(f"  Label sample     : {[CIFAR10_CLASSES[l] for l in labels[:5].tolist()]}")
print("=" * 40)

     CIFAR-10 Dataset Ready!
  Training samples : 50,000
  Test samples     : 10,000
  Classes          : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
----------------------------------------
  Batch shape      : torch.Size([128, 3, 32, 32])
  Pixel min        : -2.4291
  Pixel max        : 2.7537
  Label sample     : ['frog', 'airplane', 'deer', 'automobile', 'bird']


In [14]:
class ResNet18_CIFAR10(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = models.resnet18(weights=None)  # no pretrained weights

        # Replace first conv: smaller kernel + no stride + no maxpool
        # Original: 7x7, stride=2 → too aggressive for 32x32 images
        self.model.conv1 = nn.Conv2d(
            in_channels=3, out_channels=64,
            kernel_size=3, stride=1, padding=1, bias=False
        )
        # Remove maxpool (would shrink 32x32 too aggressively)
        self.model.maxpool = nn.Identity()

        # Replace final FC layer for 10 CIFAR-10 classes
        self.model.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        return self.model(x)

In [15]:
model = ResNet18_CIFAR10(num_classes=10).to(device)

In [16]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 11,173,962


In [17]:
NUM_EPOCHS = 100

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True          # Nesterov momentum for faster convergence
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,      # Decay LR over full training cycle
    eta_min=1e-4           # Minimum LR floor
)

In [18]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct      += predicted.eq(labels).sum().item()
        total        += labels.size(0)

    return total_loss / total, 100.0 * correct / total

In [19]:
# ── Validation Loop ───────────────────────────────────────────────────
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct      += predicted.eq(labels).sum().item()
            total        += labels.size(0)

    return total_loss / total, 100.0 * correct / total

In [20]:
best_acc        = 0.0
best_weights    = None
history         = {"train_loss": [], "train_acc": [],
                   "val_loss":   [], "val_acc":   [],
                   "lr":         []}

print("=" * 65)
print(f"  Training ResNet-18 on CIFAR-10 for {NUM_EPOCHS} epochs")
print(f"  Device: {device}")
print("=" * 65)
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | "
      f"{'Val Loss':>8} | {'Val Acc':>8} | {'LR':>8}")
print("-" * 65)


  Training ResNet-18 on CIFAR-10 for 100 epochs
  Device: mps
 Epoch | Train Loss | Train Acc | Val Loss |  Val Acc |       LR
-----------------------------------------------------------------


In [21]:
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    current_lr = scheduler.get_last_lr()[0]

    # Log history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    # Save best weights
    if val_acc > best_acc:
        best_acc     = val_acc
        best_weights = copy.deepcopy(model.state_dict())
        tag = "✅ best"
    else:
        tag = ""

    elapsed = time.time() - t0
    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2f}% | "
          f"{val_loss:>8.4f} | {val_acc:>8.2f}% | {current_lr:>8.6f}  "
          f"{tag}  ({elapsed:.1f}s)")

     1 |     1.9068 |    31.25% |   1.5124 |    45.33% | 0.099975  ✅ best  (134.9s)
     2 |     1.3589 |    50.43% |   1.2179 |    56.02% | 0.099901  ✅ best  (124.0s)
     3 |     1.0674 |    61.79% |   0.9677 |    65.98% | 0.099778  ✅ best  (122.3s)
     4 |     0.8919 |    68.40% |   0.8376 |    70.33% | 0.099606  ✅ best  (122.2s)
     5 |     0.7485 |    73.75% |   0.8887 |    70.89% | 0.099385  ✅ best  (122.1s)
     6 |     0.6485 |    77.49% |   0.6801 |    77.00% | 0.099115  ✅ best  (122.9s)
     7 |     0.5809 |    79.94% |   0.7533 |    74.58% | 0.098797    (120.4s)
     8 |     0.5356 |    81.63% |   0.6039 |    80.08% | 0.098431  ✅ best  (120.4s)
     9 |     0.5036 |    82.58% |   0.5267 |    81.90% | 0.098017  ✅ best  (112.8s)
    10 |     0.4779 |    83.76% |   0.7558 |    75.19% | 0.097555    (90.3s)
    11 |     0.4624 |    84.14% |   0.5941 |    80.25% | 0.097047    (91.8s)
    12 |     0.4432 |    84.55% |   0.5915 |    80.90% | 0.096492    (91.9s)
    13 |     0.4303

In [22]:
# ── Save best model to disk ───────────────────────────────────────────
torch.save(best_weights, "resnet18_cifar10_clean.pth")
model.load_state_dict(best_weights)   # reload best weights into model

print("=" * 65)
print(f"  Training complete!")
print(f"  Best validation accuracy : {best_acc:.2f}%")
print(f"  Weights saved to         : resnet18_cifar10_clean.pth")
print("=" * 65)

  Training complete!
  Best validation accuracy : 94.95%
  Weights saved to         : resnet18_cifar10_clean.pth


In [37]:
import numpy as np
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from art.estimators.classification import PyTorchClassifier

In [38]:
art_classifier = PyTorchClassifier(
    model=model,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(3, 32, 32),
    nb_classes=10,
    clip_values=(0.0, 1.0),        # pixel range BEFORE normalization
    device_type=art_device         # "cpu" for MPS, "gpu" for CUDA
)

In [39]:
# ── Metric 1: Clean Accuracy ──────────────────────────────────────────
def clean_accuracy(model, loader, device):
    """Top-1 accuracy on the original unperturbed test set."""
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            _, predicted   = outputs.max(1)
            correct        += predicted.eq(labels).sum().item()
            total          += labels.size(0)

    return 100.0 * correct / total

In [40]:
# ── Metric 2: Adversarial Accuracy ───────────────────────────────────
def adversarial_accuracy(art_classifier, loader, attack, device, n_batches=None):
    """
    Accuracy after adversarial perturbation.
    n_batches: limit evaluation to first N batches (None = full test set)
               PGD is slow — use n_batches=10 for quick checks
    """
    correct, total = 0, 0

    for i, (images, labels) in enumerate(loader):
        if n_batches and i >= n_batches:
            break

        # ART expects numpy arrays in [0, 1] — undo normalization first
        images_np = denormalize(images).numpy()           # (B, 3, 32, 32)
        labels_np = labels.numpy()

        # Generate adversarial examples (returns numpy)
        images_adv = attack.generate(x=images_np)

        # Predict on adversarial examples
        preds = art_classifier.predict(images_adv)        # (B, 10) logits
        predicted = np.argmax(preds, axis=1)

        correct += (predicted == labels_np).sum()
        total   += labels_np.shape[0]

        if (i + 1) % 10 == 0:
            print(f"  Evaluated {total} samples...", end="\r")

    return 100.0 * correct / total

In [41]:
# ── Helper: Denormalize for ART ───────────────────────────────────────
# ART clip_values=(0,1) expects raw [0,1] pixels, not standardized tensors
# We must undo the CIFAR-10 mean/std normalization before passing to ART

def denormalize(tensor):
    """Undo CIFAR-10 normalization → returns tensor in [0, 1]."""
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std  = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0.0, 1.0)


In [42]:
# ── Metric 3: Robustness Gap ──────────────────────────────────────────
def robustness_gap(clean_acc, adv_acc):
    """
    Gap between clean and adversarial accuracy.
    Smaller = more robust. Goal: minimize this in later phases.
    """
    return clean_acc - adv_acc

In [43]:
# ── Define Attacks ────────────────────────────────────────────────────
# FGSM — single-step, epsilon=8/255 (standard CIFAR-10 benchmark)
fgsm_attack = FastGradientMethod(
    estimator=art_classifier,
    eps=8/255,              # perturbation budget
    eps_step=8/255,         # same as eps for single-step
    batch_size=128
)

# PGD — iterative, 40 steps (strong attack, use n_batches to limit eval time)
pgd_attack = ProjectedGradientDescent(
    estimator=art_classifier,
    eps=8/255,              # same budget as FGSM for fair comparison
    eps_step=2/255,         # step size per iteration
    max_iter=40,            # 40 iterations = strong attack
    num_random_init=1,      # random start for stronger evaluation
    batch_size=128
)

In [45]:
# ── Full Clean Reset Cell ─────────────────────────────────────────────
import torch, copy
import torch.nn as nn
import torch.optim as optim

# Step 1: Confirm device
print(f"Device: {device}")  # should print 'mps'

# Step 2: Re-instantiate model and EXPLICITLY move to MPS
model = ResNet18_CIFAR10(num_classes=10)
model = model.to(device)

# Step 3: Confirm ALL parameters are on MPS
param_devices = set(str(p.device) for p in model.parameters())
buffer_devices = set(str(b.device) for b in model.buffers())
print(f"Parameter devices : {param_devices}")   # must show {'mps:0'}
print(f"Buffer devices    : {buffer_devices}")   # must show {'mps:0'}

# Step 4: Load saved best weights if they exist
import os
if os.path.exists("resnet18_cifar10_clean.pth"):
    state_dict = torch.load("resnet18_cifar10_clean.pth", map_location=device)
    model.load_state_dict(state_dict)
    print("✅ Loaded best weights from disk")
else:
    print("⚠️  No saved weights found — using current model state")

# Step 5: Re-confirm device after loading weights
param_devices = set(str(p.device) for p in model.parameters())
print(f"Post-load parameter devices: {param_devices}")  # still must be {'mps:0'}

# Step 6: Re-define criterion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=0.1, momentum=0.9,
    weight_decay=5e-4, nesterov=True
)

# Step 7: CPU copy for ART (MPS not supported by ART)
model_cpu = copy.deepcopy(model).to("cpu")
param_devices_cpu = set(str(p.device) for p in model_cpu.parameters())
print(f"ART model devices : {param_devices_cpu}")  # must show {'cpu'}

# Step 8: Rebuild ART classifier on CPU copy
from art.estimators.classification import PyTorchClassifier

criterion_cpu = nn.CrossEntropyLoss()
optimizer_cpu = optim.SGD(model_cpu.parameters(), lr=0.1, momentum=0.9)

art_classifier = PyTorchClassifier(
    model=model_cpu,
    loss=criterion_cpu,
    optimizer=optimizer_cpu,
    input_shape=(3, 32, 32),
    nb_classes=10,
    clip_values=(0.0, 1.0),
    device_type="cpu"
)

print("\n✅ All components reset and verified!")
print(f"   model       → {next(model.parameters()).device}")
print(f"   model_cpu   → {next(model_cpu.parameters()).device}")
print(f"   art_device  → cpu")

Device: mps
Parameter devices : {'mps:0'}
Buffer devices    : {'mps:0'}
✅ Loaded best weights from disk
Post-load parameter devices: {'mps:0'}
ART model devices : {'cpu'}

✅ All components reset and verified!
   model       → mps:0
   model_cpu   → cpu
   art_device  → cpu


In [48]:
# ── Safe Evaluation ───────────────────────────────────────────────────

# Quick tensor test before full eval
dummy = torch.ones(1, 3, 32, 32).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"✅ Forward pass test passed — output shape: {out.shape}")

# Now run metrics
print("\n[1/3] Computing clean accuracy...")
acc_clean = clean_accuracy(model, test_loader, device)
print(f"      Clean Accuracy : {acc_clean:.2f}%")

print("\n[2/3] Running FGSM attack...")
acc_fgsm = adversarial_accuracy(art_classifier, test_loader, fgsm_attack, device)
gap_fgsm = robustness_gap(acc_clean, acc_fgsm)
print(f"      FGSM Accuracy  : {acc_fgsm:.2f}%")
print(f"      Robustness Gap : {gap_fgsm:.2f}%")

# ── Faster PGD Evaluation with Progress Tracking ─────────────────────

from art.attacks.evasion import ProjectedGradientDescent
import time

# Option A: Fewer iterations for quick check (still strong attack)
pgd_attack_fast = ProjectedGradientDescent(
    estimator=art_classifier,
    eps=8/255,
    eps_step=2/255,
    max_iter=10,           # ← reduced from 40 to 10 for speed
    num_random_init=1,
    batch_size=128,
    verbose=True           # shows tqdm progress bar per batch
)

# Option B: Reduce to 5 batches instead of 10
N_BATCHES = 5              # ← ~640 samples, faster estimate

print("[3/3] Running PGD attack (fast version)...")
print(f"      Config: 10 iterations, {N_BATCHES} batches (~{N_BATCHES*128} samples)")
print(f"      Estimated time: ~2–5 mins on CPU\n")

t0 = time.time()

correct, total = 0, 0
for i, (images, labels) in enumerate(test_loader):
    if i >= N_BATCHES:
        break

    images_np = denormalize(images).numpy()
    labels_np = labels.numpy()

    batch_start = time.time()
    images_adv  = pgd_attack_fast.generate(x=images_np)
    batch_time  = time.time() - batch_start

    preds     = art_classifier.predict(images_adv)
    predicted = np.argmax(preds, axis=1)
    correct   += (predicted == labels_np).sum()
    total     += labels_np.shape[0]

    batch_acc = 100.0 * (predicted == labels_np).mean()
    elapsed   = time.time() - t0
    eta       = (elapsed / (i + 1)) * (N_BATCHES - i - 1)

    print(f"  Batch {i+1}/{N_BATCHES} | "
          f"Batch Acc: {batch_acc:.1f}% | "
          f"Elapsed: {elapsed:.0f}s | "
          f"ETA: {eta:.0f}s")

acc_pgd  = 100.0 * correct / total
gap_pgd  = robustness_gap(acc_clean, acc_pgd)

print(f"\n{'=' * 45}")
print(f"  Clean Accuracy     : {acc_clean:.2f}%")
print(f"  FGSM Accuracy      : {acc_fgsm:.2f}%  (gap: {gap_fgsm:.2f}%)")
print(f"  PGD  Accuracy      : {acc_pgd:.2f}%  (gap: {gap_pgd:.2f}%)")
print(f"{'=' * 45}")

baseline_metrics = {
    "clean_acc" : acc_clean,
    "fgsm_acc"  : acc_fgsm,
    "fgsm_gap"  : gap_fgsm,
    "pgd_acc"   : acc_pgd,
    "pgd_gap"   : gap_pgd
}
print("\n✅ baseline_metrics saved for Phase 2 comparison")

✅ Forward pass test passed — output shape: torch.Size([1, 10])

[1/3] Computing clean accuracy...
      Clean Accuracy : 94.58%

[2/3] Running FGSM attack...
      FGSM Accuracy  : 19.30%
      Robustness Gap : 75.28%
[3/3] Running PGD attack (fast version)...
      Config: 10 iterations, 5 batches (~640 samples)
      Estimated time: ~2–5 mins on CPU



PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Batch 1/5 | Batch Acc: 15.6% | Elapsed: 18s | ETA: 73s


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Batch 2/5 | Batch Acc: 18.8% | Elapsed: 37s | ETA: 56s


...

  Batch 3/5 | Batch Acc: 15.6% | Elapsed: 55s | ETA: 37s


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Batch 4/5 | Batch Acc: 15.6% | Elapsed: 73s | ETA: 18s


PGD - Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Batch 5/5 | Batch Acc: 20.3% | Elapsed: 91s | ETA: 0s

  Clean Accuracy     : 94.58%
  FGSM Accuracy      : 19.30%  (gap: 75.28%)
  PGD  Accuracy      : 17.19%  (gap: 77.39%)

✅ baseline_metrics saved for Phase 2 comparison
